In [1]:
# Raw Data Loading
import pandas as pd

red_url = 'https://raw.githubusercontent.com/PinkWink/ML_tutorial' +\
                        '/master/dataset/winequality-red.csv'
white_url = 'https://raw.githubusercontent.com/PinkWink/ML_tutorial' +\
                        '/master/dataset/winequality-white.csv'
    
red_wine = pd.read_csv(red_url, sep = ';')
white_wine = pd.read_csv(white_url, sep = ";")

red_wine['color'] = 1
white_wine['color']= 0

wine = pd.concat([red_wine, white_wine])

In [2]:
# taste 컬럼 추가
wine['taste'] = [1 if grade > 5 else 0 for grade in wine['quality']]

In [3]:
# with out Cross Validation
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

x = wine.drop(['taste', 'quality'], axis=1)
y = wine['taste']

x_train, x_test, y_train, y_test = train_test_split(x,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=13)

dtc = DecisionTreeClassifier(max_depth=2, random_state=13)
dtc.fit(x_train, y_train)

pred_train = dtc.predict(x_train)
pred_test = dtc.predict(x_test)

print(f'Train Acc: {accuracy_score(y_train, pred_train)}')
print(f'Test Acc: {accuracy_score(y_test, pred_test)}')

Train Acc: 0.7294593034442948
Test Acc: 0.7161538461538461


In [4]:
# 좀 더 객관적인 평가를 위해 Cross Validation 실행
from sklearn.model_selection import KFold

kfold = KFold(n_splits=5)
dtc_cv = DecisionTreeClassifier(max_depth=2, random_state=13)

for train_idx, test_idx in kfold.split(x):
    print(len(train_idx), len(test_idx))

5197 1300
5197 1300
5198 1299
5198 1299
5198 1299


In [ ]:
# for문을 통해 학습
# 5번의 교차 검증을 통해 60% ~ 79% 까지 정확도가 분포해 있음
cv_accuracy = []
for train_idx, test_idx in kfold.split(x, y):
    x_train, x_test = x.iloc[train_idx], x.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    dtc_cv.fit(x_train, y_train)
    pred = dtc_cv.predict(x_test)
    cv_accuracy.append(accuracy_score(y_test, pred))

cv_accuracy

[0.6007692307692307,
 0.6884615384615385,
 0.7090069284064665,
 0.7628945342571208,
 0.7867590454195535]

In [7]:
# 정확도의 평균 (편차가 클수록 평균 의미 x)
import numpy as np

np.mean(cv_accuracy)

0.709578255462782

In [10]:
# 현재 와인 데이터는 레드와인과 화이트와인의 비율이 맞지 않음
# -> StratifiedKFold 사용
from sklearn.model_selection import StratifiedKFold

skfold = StratifiedKFold(n_splits=5)
dtc_cv = DecisionTreeClassifier(max_depth=2, random_state=13)

cv_accuracy = []
for train_idx, test_idx in skfold.split(x, y):
    x_train, x_test = x.iloc[train_idx], x.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    dtc_cv.fit(x_train, y_train)
    pred = dtc_cv.predict(x_test)
    cv_accuracy.append(accuracy_score(y_test, pred))

cv_accuracy

[0.5523076923076923,
 0.6884615384615385,
 0.7143956889915319,
 0.7321016166281755,
 0.7567359507313318]

In [11]:
np.mean(cv_accuracy)

0.6888004974240539

In [12]:
# for문을 사용하지 않고 간단하게
from sklearn.model_selection import cross_val_score

skfold = StratifiedKFold(n_splits=5)
dtc_cv = DecisionTreeClassifier(max_depth=2, random_state=13)

cross_val_score(dtc_cv, x, y, scoring=None, cv=skfold)

array([0.55230769, 0.68846154, 0.71439569, 0.73210162, 0.75673595])

In [13]:
# max_depth를 5로
dtc_cv = DecisionTreeClassifier(max_depth=5, random_state=13)

cross_val_score(dtc_cv, x, y, scoring=None, cv=skfold)

array([0.50076923, 0.62615385, 0.69745958, 0.7582756 , 0.74903772])

In [14]:
from sklearn.model_selection import cross_validate

cross_validate(dtc_cv, x, y, scoring=None, cv=skfold, return_train_score=True)

{'fit_time': array([0.02000713, 0.01651597, 0.02299881, 0.01799893, 0.01399589]),
 'score_time': array([0.00199366, 0.00200629, 0.00300217, 0.0010078 , 0.00100112]),
 'test_score': array([0.50076923, 0.62615385, 0.69745958, 0.7582756 , 0.74903772]),
 'train_score': array([0.78795459, 0.78045026, 0.77568295, 0.76356291, 0.76279338])}